<a href="https://colab.research.google.com/github/AISHIK1/NN-with-Pytorch/blob/main/fanshion_mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import torch
import cv2
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader,Dataset
from sklearn.model_selection import train_test_split
import torch.nn as nn
from torchinfo import summary

ModuleNotFoundError: No module named 'torchinfo'

In [2]:
import os

os.environ['KAGGLE_USERNAME'] = 'aishik996'
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_61236d6a08ac73ba31896583260ad688'



!kaggle datasets download -d zalando-research/fashionmnist

Dataset URL: https://www.kaggle.com/datasets/zalando-research/fashionmnist
License(s): other
100% 68.8M/68.8M [00:02<00:00, 29.6MB/s]



In [3]:
import zipfile

with zipfile.ZipFile('/content/fashionmnist.zip', 'r') as zip_ref:
    zip_ref.extractall()

In [4]:
df_train=pd.read_csv('/content/fashion-mnist_train.csv')

In [5]:
df_test=pd.read_csv('/content/fashion-mnist_test.csv')

In [6]:
df_test.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,9,8,...,103,87,56,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,14,53,99,...,0,0,0,0,63,53,31,0,0,0
3,2,0,0,0,0,0,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
df=pd.concat([df_train,df_test],axis=0)

In [8]:
df_train.shape

(60000, 785)

In [9]:
df_test.shape

(10000, 785)

In [10]:
df.shape

(70000, 785)

In [11]:
y=df.iloc[:,0].values

In [12]:
X=df.iloc[:,1:].values

In [13]:
#standerize
X=X/255.0

In [14]:
X

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.00392157,
        0.        ],
       [0.        , 0.00392157, 0.01176471, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]])

In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [16]:
#creating custom dataset

class mydataset(Dataset):

  def __init__(self,features,labels):

    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)


  def __len__(self):

    return self.features.shape[0]

  def __getitem__(self,index):

    return self.features[index],self.labels[index]

In [17]:
train_dataset=mydataset(X_train,y_train)
test_dataset=mydataset(X_test,y_test)

In [18]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [19]:
from torch.nn.modules.activation import Softmax
#define the nn class

class Model(nn.Module):

  def __init__(self,num_features,num_labels):

    super().__init__()

    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,num_labels),
        nn.Softmax()
        )


  def forward(self,features):

    out=self.model(features)

    return out

In [20]:
learning_rate=0.1
epochs=50

model=Model(X_train.shape[1],10)

optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

loss_function=nn.CrossEntropyLoss()

In [22]:
for epoch in range(epochs):

  total_loss=0

  for batch_features,batch_label in train_loader:
    y_pred=model(batch_features)

    loss=loss_function(y_pred,batch_label)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_loss+=loss.item()

  avg_loss=total_loss/len(batch_features)
  print(f'epochs : {epoch+1}  avg_loss : {avg_loss}')

epochs : 1  avg_loss : 101.43396140635014
epochs : 2  avg_loss : 92.01822138950229
epochs : 3  avg_loss : 91.2268595136702
epochs : 4  avg_loss : 90.73468926548958
epochs : 5  avg_loss : 90.45650730282068
epochs : 6  avg_loss : 90.22857950255275
epochs : 7  avg_loss : 90.10852063074708
epochs : 8  avg_loss : 89.96617998182774
epochs : 9  avg_loss : 89.79193870350718
epochs : 10  avg_loss : 89.69300596415997
epochs : 11  avg_loss : 89.55577913299203
epochs : 12  avg_loss : 89.51767360791564
epochs : 13  avg_loss : 89.37734054028988
epochs : 14  avg_loss : 89.27454521507025
epochs : 15  avg_loss : 89.20351871103048
epochs : 16  avg_loss : 89.09702949225903
epochs : 17  avg_loss : 89.05360227823257
epochs : 18  avg_loss : 88.96862327307463
epochs : 19  avg_loss : 88.99860620498657
epochs : 20  avg_loss : 88.88158732280135
epochs : 21  avg_loss : 88.84600192308426
epochs : 22  avg_loss : 88.78822444751859
epochs : 23  avg_loss : 88.72244617342949
epochs : 24  avg_loss : 88.69594988971949
e

In [23]:
model.eval()

Model(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
    (5): Softmax(dim=None)
  )
)

In [41]:
total=0

correct=0

with torch.no_grad():

  for batch_feature,batch_label in train_loader:

    outputs=model(batch_feature)

    _,predicted=torch.max(outputs,1)

    total+=batch_label.shape[0]

    correct+=(predicted==batch_label).sum().item()

  accuracy=correct/total

  print(f'accuracy : {accuracy}')

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


accuracy : 0.8568214285714286


In [37]:
predicted[1]

tensor([7, 0, 9, 2, 4, 0, 4, 0, 1, 5, 8, 5, 0, 1, 4, 2, 9, 7, 0, 1, 9, 9, 0, 3,
        0, 8, 9, 0, 0, 2, 4, 3])

In [38]:
predicted[0]

tensor([1.0000, 1.0000, 1.0000, 0.9960, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 0.8359, 1.0000, 1.0000, 1.0000])

In [40]:
predicted

torch.return_types.max(
values=tensor([1.0000, 1.0000, 1.0000, 0.9960, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 0.8359, 1.0000, 1.0000, 1.0000]),
indices=tensor([7, 0, 9, 2, 4, 0, 4, 0, 1, 5, 8, 5, 0, 1, 4, 2, 9, 7, 0, 1, 9, 9, 0, 3,
        0, 8, 9, 0, 0, 2, 4, 3]))

In [42]:
total=0

correct=0

with torch.no_grad():

  for batch_feature,batch_label in train_loader:

    outputs=model(batch_feature)

    predicted=torch.max(outputs,1)

    total+=batch_label.shape[0]

    correct+=(predicted[1]==batch_label).sum().item()

  accuracy=correct/total

  print(f'accuracy : {accuracy}')

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


accuracy : 0.8568214285714286
